# Trabajo Práctico N.º 3 — Recomendador personalizado de libros

**Procesamiento de Lenguaje Natural · TUIA · UNR**

**Integrantes:** Josías Calabozo · Sharo Giuntoli · Ismael Darruiz · Sebastián Di Carlo

**Datos de la cátedra:** `data/catalogo_tp3.jsonl` (8.747 libros, algunos contaminados) y `data/muestra_etiquetada.csv` (500 libros del mismo catálogo con la columna `contaminado`).

El notebook sigue los ocho pasos del enunciado, con sus mismos títulos:

| Paso | Temas de la unidad |
|---|---|
| 1. Cargar y revisar los libros | Limpieza · semejanza de cadenas |
| 2. Auditar el catálogo | Expresiones regulares · detección de idioma · NER · clasificación · similitud |
| 3. Registrar las preferencias de un lector | — |
| 4. Separar los libros para aprender y evaluar | Entrenamiento y prueba |
| 5. Entrenar dos clasificadores | Clasificación con TF-IDF y embeddings |
| 6. Comprobar las predicciones | Métricas · validación cruzada |
| 7. Convertir las predicciones en recomendaciones | Similitud del coseno · precision@5 |
| 8. Recomendar libros del catálogo completo | Resumen extractivo · frases sustantivas y entidades |

**Cómo ejecutarlo:** ver el [README](README.md) (entorno, modelo de spaCy y planilla de preferencias).

---
# Preparación

In [1]:
import os
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import spacy
from langdetect import DetectorFactory, detect, detect_langs
from langdetect.lang_detect_exception import LangDetectException
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, precision_recall_curve,
                             precision_recall_fscore_support, precision_score, recall_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import (GroupKFold, RepeatedStratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from thefuzz import fuzz

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # aviso de la caché de Hugging Face en Windows
pd.set_option("display.max_colwidth", None)   # las tablas muestran los textos completos, sin cortarlos

# Semilla de todo lo que tiene azar (divisiones, mezcla del catálogo): con ella, cada corrida da lo mismo
SEMILLA = 42
DetectorFactory.seed = 0   # langdetect también tiene una parte aleatoria

# Rutas relativas a P3/, la carpeta del notebook (ver el README)
RUTA_CATALOGO = "data/catalogo_tp3.jsonl"
RUTA_MUESTRA = "data/muestra_etiquetada.csv"
RUTA_LIMPIO = "data/catalogo_limpio.csv"
GRUPO = "grupoNN"   # reemplazar NN por el número de grupo
RUTA_SOSPECHOSOS = f"data/{GRUPO}_sospechosos.csv"
RUTA_PREFERENCIAS = f"data/{GRUPO}_preferencias.csv"

C:\Users\PC\Documents\tuia\tuia-nlp\PLN_TUIA\P3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
# Paso 1. Cargar y revisar los libros

Conservamos `slug` (identificador), `title` (para mostrar), `authors` y `synopsis` (texto de entrada).

In [2]:
catalogo = pd.read_json(RUTA_CATALOGO, lines=True)
print(f"Catálogo de la cátedra: {len(catalogo)} libros")

libros = catalogo[["slug", "title", "authors", "synopsis"]].copy()

# Las palabras en cursiva quedaron separadas por saltos de línea. Cada secuencia de espacios en blanco
# pasa a ser un solo espacio. No se borra ningún otro carácter: algunos son evidencia en el paso 2
antes = libros.loc[libros["synopsis"].str.contains("El primer americano\n", regex=False, na=False), "synopsis"].iloc[0]
libros["synopsis"] = libros["synopsis"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
print("Antes:  ", repr(antes[:60]))
print("Después:", repr(re.sub(r"\s+", " ", antes)[:60]))

eliminados = {}   # motivo → cantidad de libros eliminados, para el resumen del final del paso

sin_sinopsis = libros["synopsis"] == ""
eliminados["sin sinopsis"] = sin_sinopsis.sum()
libros = libros[~sin_sinopsis]

# duplicated marca la segunda aparición (y las siguientes) de cada slug
slug_repetido = libros["slug"].duplicated()
eliminados["slug repetido"] = slug_repetido.sum()
libros = libros[~slug_repetido]
print(eliminados)

Catálogo de la cátedra: 8747 libros


Antes:   'El primer americano\n, ambiciosa obra que lleva camino de con'
Después: 'El primer americano , ambiciosa obra que lleva camino de con'
{'sin sinopsis': np.int64(1), 'slug repetido': np.int64(0)}


## Duplicados exactos de sinopsis

Antes de borrarlos, miramos qué libros comparten exactamente la misma sinopsis.

In [3]:
# keep=False marca todas las apariciones de una sinopsis repetida, también la primera
repetidas = libros[libros["synopsis"].duplicated(keep=False)]
grupos = repetidas.groupby("synopsis").agg(libros=("slug", "size"),
                                           autores=("authors", lambda a: a.str.join(", ").nunique()),
                                           titulos=("title", lambda t: " | ".join(t.head(3))))
print(f"{len(grupos)} sinopsis aparecen en más de un libro ({len(repetidas)} libros en total)")
grupos.sort_values("libros", ascending=False).reset_index(drop=True).head(8)

33 sinopsis aparecen en más de un libro (131 libros en total)


,libros,autores,titulos
0,58,1,Alemán – Unidad 40 | Alemán – Unidad 15 | Alemán – Unidad 13
1,7,7,Nueva Dimensión 7 | Nueva Dimensión 5 | Nueva Dimensión 1
2,4,1,Teatro crítico universal VII | Teatro crítico universal VI | Teatro crítico universal VIII
3,3,1,Dionisíacas Cantos XXV-XXXVI | Dionisíacas Cantos XXXVII-XLVIII | Dionisíacas Cantos XIII-XXIV
4,3,1,El fin de una cuadrilla | El santón de la montaña | La pista de un bandido
5,2,1,Prácticas sexuales alternativas (I) | Prácticas sexuales alternativas (II)
6,2,1,Los años irreverentes. Escritos completos en El Amante/Cine Vol.2 | Los años irreverentes. Escritos completos en El Amante/Cine Vol.1
7,2,1,Comunicaciones unificadas con Elastix | Comunicaciones unificadas con Elastix (Volumen 2)


In [4]:
# Conservamos un libro por sinopsis: el primero que aparece en el catálogo
sinopsis_repetida = libros["synopsis"].duplicated()
eliminados["sinopsis repetida"] = sinopsis_repetida.sum()
libros = libros[~sinopsis_repetida].reset_index(drop=True)
print(f"Eliminados por sinopsis repetida: {eliminados['sinopsis repetida']}")

Eliminados por sinopsis repetida: 98


**Qué muestra**

- **33 sinopsis se repiten en 131 libros.** El caso más grande es un curso: las 58 unidades de «Alemán» comparten la misma sinopsis. También hay revistas (7 números de *Nueva Dimensión*, cada uno con un autor distinto) y obras en varios tomos (*Teatro crítico universal*, *Dionisíacas*).
- Son libros distintos con la misma descripción: para un recomendador que lee sinopsis, son indistinguibles. Conservamos el primero de cada grupo y **eliminamos 98**.

## Versiones casi iguales de una obra

Buscamos libros con los mismos autores y un título muy parecido, por ejemplo con o sin subtítulo. Para no comparar todos contra todos, comparamos solo los libros de los mismos autores.

**Criterio.** La semejanza de cadenas mide la forma, no el significado, así que combinamos tres señales:

1. **Título:** `fuzz.token_set_ratio` de `thefuzz` (Unidad 3) sobre los títulos normalizados. Compara los conjuntos de palabras y da 100 cuando todas las palabras de un título están en el otro, como en «Freakonomics» y «Freakonomics (edición ampliada)». Antes de comparar quitamos lo que está entre paréntesis, que suele ser la edición o la traducción.
2. **Números:** los dos títulos tienen que tener los mismos números, arábigos o romanos. Así «Tratados hipocráticos II» y «Tratados hipocráticos III» quedan separados, como «Conversaciones con Dios I» y «II» en el ejemplo del enunciado.
3. **Sinopsis:** similitud del coseno con TF-IDF de al menos 0,5. Dos obras distintas pueden tener un título contenido en el otro («Drácula» y «El invitado de Drácula»), pero no la misma descripción.

Unimos un par solo si cumple las tres. Ante la duda conservamos los dos libros: borrar un libro legítimo es peor que dejar dos versiones.

In [5]:
NUMEROS = re.compile(r"\b(?:\d+|[ivx]+)\b")   # números arábigos o romanos: "2", "1937", "iii"


def normalizar_titulo(titulo):
    """Minúsculas, sin lo que está entre paréntesis y sin puntuación."""
    titulo = titulo.lower()
    titulo = re.sub(r"\(.*?\)", " ", titulo)   # "Hamlet (Ed. bilingüe)" → "hamlet "
    titulo = re.sub(r"[^\w\s]", " ", titulo)   # "Ella. Hija de la sabiduría" → "ella  hija de la sabiduría"
    return re.sub(r"\s+", " ", titulo).strip()


def mismo_titulo(titulo_a, titulo_b):
    """True si un título contiene todas las palabras del otro y los dos tienen los mismos números."""
    # Los números se buscan en el título completo, también entre paréntesis: "(Tomo I)" no es "(Tomo II)"
    mismos_numeros = NUMEROS.findall(titulo_a.lower()) == NUMEROS.findall(titulo_b.lower())
    return mismos_numeros and fuzz.token_set_ratio(normalizar_titulo(titulo_a), normalizar_titulo(titulo_b)) == 100


# Matriz TF-IDF de las sinopsis, para comparar las de cada par. Sus filas siguen el orden de libros
X_sinopsis = TfidfVectorizer().fit_transform(libros["synopsis"])

filas = []
# Un grupo por cada lista de autores: solo se comparan libros de los mismos autores
for autores, grupo in libros.groupby(libros["authors"].str.join(", ")):
    indices = grupo.index.tolist()
    for a in range(len(indices)):
        for b in range(a + 1, len(indices)):   # b empieza en a + 1: cada par una sola vez
            i, j = indices[a], indices[b]
            parecido = fuzz.token_set_ratio(normalizar_titulo(libros.at[i, "title"]), normalizar_titulo(libros.at[j, "title"]))
            if parecido >= 85:   # solo los pares con títulos muy parecidos son candidatos
                filas.append({"i": i, "j": j, "titulo_1": libros.at[i, "title"], "titulo_2": libros.at[j, "title"],
                              "parecido_titulo": parecido,
                              "mismo_titulo": mismo_titulo(libros.at[i, "title"], libros.at[j, "title"]),
                              "similitud_sinopsis": cosine_similarity(X_sinopsis[i], X_sinopsis[j])[0, 0].round(2)})
pares = pd.DataFrame(filas)
pares["unir"] = pares["mismo_titulo"] & (pares["similitud_sinopsis"] >= 0.5)
print(f"Pares candidatos: {len(pares)}; se unen: {pares['unir'].sum()}")

Pares candidatos: 363; se unen: 23


In [6]:
# Dos pares que unimos y dos que separamos (como conjuntos, porque el orden del par depende del catálogo)
EJEMPLOS = [{"La venganza de don Mendo (ed. Salvador García Castañeda)", "La venganza de don Mendo"},
            {"El príncipe y el mendigo", "Príncipe y mendigo"},
            {"Tratados hipocráticos II", "Tratados hipocráticos III"},
            {"Drácula", "El invitado de Drácula"}]
es_ejemplo = pares.apply(lambda par: {par["titulo_1"], par["titulo_2"]} in EJEMPLOS, axis=1)
pares.loc[es_ejemplo, ["titulo_1", "titulo_2", "parecido_titulo", "mismo_titulo", "similitud_sinopsis", "unir"]]

,titulo_1,titulo_2,parecido_titulo,mismo_titulo,similitud_sinopsis,unir
33,El invitado de Drácula,Drácula,100,True,0.27,False
135,Tratados hipocráticos III,Tratados hipocráticos II,98,False,0.87,False
207,El príncipe y el mendigo,Príncipe y mendigo,100,True,1.00,True
231,La venganza de don Mendo (ed. Salvador García Castañeda),La venganza de don Mendo,100,True,0.69,True


In [7]:
# De cada par unido se conserva el libro que aparece primero en el catálogo (i) y se elimina el otro (j)
otra_version = set(pares.loc[pares["unir"], "j"])
eliminados["otra versión de la misma obra"] = len(otra_version)
libros = libros.drop(index=list(otra_version)).reset_index(drop=True)

print(f"Libros disponibles: {len(libros)}\n")
print("Eliminados por motivo:")
print(pd.Series(eliminados).to_string())
print(f"Total eliminados: {sum(eliminados.values())}")
libros.head(5)

Libros disponibles: 8625

Eliminados por motivo:
sin sinopsis                      1
slug repetido                     0
sinopsis repetida                98
otra versión de la misma obra    23
Total eliminados: 122


,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos. Eudald Carbonell nos ayuda a visualizarlos y a entenderlos para que podamos afrontar el futuro con seguridad. La humanidad se transforma a pasos de gigante. Hechos que hasta ahora parecían ciencia ficción están a la orden del día: inteligencia artificial, nuevos órdenes sociales, viajes a otros planetas… Los humanos estamos constantemente construyendo escenarios y realidades que pueden parecer inconmensurables: ni siquiera la ciencia ficción puede llegar a adivinar lo que podemos llegar a imaginar y producir. Ante este mundo de posibilidades, Eudald Carbonell tiene claro que la historia, el conocimiento, la ciencia, la tecnología y el pensamiento crítico socializados pueden ser los referentes que nos aseguren el buen funcionamiento y el futuro como humanos. En esta obra, el eminente pensador pone orden a todos estos debates y busca darnos la clave con rigor y claridad."
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importantes controversias sobre la prehistoria, hasta el punto de cambiar su interpretación. Hoy en día puede extraerse y analizarse ADN antiguo y, en buena medida gracias a las aportaciones del propio David Reich, se ha puesto de manifiesto que la genómica es tan relevante como la arqueología o la lingüística a la hora de estudiar las poblaciones del pasado. En Quiénes somos y cómo hemos llegado hasta aquí, Reich nos enseña que el genoma humano, además de aportar toda la información que necesita un óvulo humano fertilizado para desarrollarse, contiene en sí mismo la historia de nuestra especie. La revolución de la genómica y del ADN antiguo está transformando lo que sabemos sobre el linaje de los humanos y, según nos explica Reich, los estudios de ADN revelan una larga historia de desigualdad entre poblaciones distintas, así como entre ambos sexos y entre individuos de una misma población. Esta obra desmiente la interpretación ortodoxa de que no hay diferencias biológicas relevantes entre poblaciones humanas y, al mismo tiempo, utiliza los sólidos datos de la genómica para ilustrar la improbabilidad de que las diferencias, que sí existen, se correspondan con los estereotipos de siempre."
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de nuestra era, Teotihuacan dejará su influencia en diversas regiones de Mesoamérica. Fue contemporánea de ciudades tan importantes como Monte Albán, en Oaxaca; Palenque y Yaxchilán, en Chiapas, y Cholula, en Puebla. Durante siete siglos se desarrolló y fue la ciudad con mayor número de habitantes, en una extensión de 22 kilómetros cuadrados. La ciudad estaba dividida en dos ejes y en ella vemos los comienzos de toda una serie de características, tanto religiosas como de organización social, que posteriormente adoptarán las culturas que se asentaron en el centro de México. Ciudad cosmopolita, en ella se encontró presencia de otras culturas como la maya y la zapoteca. ¿Cómo surgió Teotihuacan? ¿Cuáles fueron los factores que determinaron su colapso hacia el año 750 d. C.? El arqueólogo Eduardo Matos Moctezuma nos lleva a lo largo de estas páginas por los senderos de esta urbe que marcó una serie de características que dejaron su huella en diversas sociedades mesoamericanas."
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"Entre los siglos VIII y VI a. C. los fenicios establecieron el primer sistema comercial mediterráneo, que abarcó desde el actual Líbano hasta la Península Ibérica y Marruecos. Mediante la fundación de una serie de colonias en las principales costas e islas del Mediterráneo

**Qué muestra**

- **Quedan 8.625 libros.** Eliminamos 122: 1 sin sinopsis, ninguno por slug repetido, 98 por sinopsis repetida y 23 por ser otra versión de una obra.
- **Pares que unimos:** «La venganza de don Mendo (ed. Salvador García Castañeda)» y «La venganza de don Mendo» (un título contiene al otro y las sinopsis se parecen 0,69), y «El príncipe y el mendigo» y «Príncipe y mendigo» (misma sinopsis).
- **Pares que separamos:** «Tratados hipocráticos II» y «III» tienen títulos casi iguales (98) y sinopsis muy parecidas (0,87), porque la colección repite su presentación, pero son tomos distintos: es el caso de «Conversaciones con Dios I» y «II» del enunciado. «Drácula» y «El invitado de Drácula» tienen un título contenido en el otro, pero sus sinopsis se parecen 0,27: son dos libros distintos.
- **Limitación:** el criterio es conservador. Dos ediciones de una obra con contratapas distintas, como «Romeo y Julieta» y «Romeo y Julieta (trad. Neruda)», quedan como libros separados.

---
# Paso 2. Auditar el catálogo

Construimos varios detectores, medimos cada uno y los combinamos. Dividimos la muestra etiquetada **por libro**: el 70 % para desarrollar los detectores (elegir umbrales, revisar errores) y el 30 % para medirlos. Las funciones `oraciones` y `medir` son las de la guía del enunciado.

In [8]:
muestra = pd.read_csv(RUTA_MUESTRA, keep_default_na=False)
# La misma normalización de espacios que en el catálogo, para buscar sobre el mismo texto
for columna in ["synopsis", "fragmento"]:
    muestra[columna] = muestra[columna].str.replace(r"\s+", " ", regex=True).str.strip()

muestra_desarrollo, muestra_prueba = train_test_split(muestra, test_size=0.3, random_state=SEMILLA,
                                                      stratify=muestra["contaminado"])
print(len(muestra_desarrollo), len(muestra_prueba))
print(f"Contaminados: {muestra_desarrollo['contaminado'].sum()} en desarrollo y {muestra_prueba['contaminado'].sum()} en prueba")


def oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]


def medir(nombre, marcado, datos):
    y = datos["contaminado"]
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado)
    print(f"{nombre}: precisión {p:.2f}, recall {r:.2f} ({marcado.sum()} libros marcados)")


def tabla_oraciones(datos):
    """Una fila por oración de cada sinopsis, con el slug del libro al que pertenece."""
    filas = [(slug, oracion) for slug, texto in zip(datos["slug"], datos["synopsis"]) for oracion in oraciones(texto)]
    return pd.DataFrame(filas, columns=["slug", "oracion"])


# Lo que marca cada detector en la parte de prueba, para la tabla del paso 2.7
marcas_prueba = {}

350 150
Contaminados: 70 en desarrollo y 30 en prueba


## 2.1 Expresiones regulares

Una expresión regular para enlaces, correos y teléfonos. Los fragmentos de desarrollo usan dominios terminados en `.example`, `.test` e `.invalid`: son terminaciones reservadas para ejemplos, que ningún sitio real usa. Sumamos las más comunes de los sitios reales.

In [9]:
URL = re.compile(
    r"https?://\S+"                                            # enlace con protocolo
    r"|www\.\S+"                                               # enlace que empieza con www.
    r"|\b[\w.-]+@[\w-]+\.[a-z]{2,}\b"                          # correo: ventas@tomozona.test
    r"|\b[a-z0-9-]+\.(?:com|net|org|example|test|invalid)\b"   # dominio suelto: tomoya.test
    r"|\+\d[\d -]{8,}\d",                                      # teléfono con código de país: +54 9 11 5217-8319
    re.IGNORECASE)


def detectar_regex(datos):
    return datos["synopsis"].str.contains(URL)


marcas_prueba["regex"] = detectar_regex(muestra_prueba)
medir("regex", marcas_prueba["regex"], muestra_prueba)

# Falsos positivos en desarrollo: libros limpios con algo que la regex reconoce
marcado = detectar_regex(muestra_desarrollo)
falsos = muestra_desarrollo[marcado & (muestra_desarrollo["contaminado"] == 0)]
print("\nFalsos positivos en desarrollo:")
for titulo, texto in zip(falsos["title"], falsos["synopsis"]):
    print(f"  {titulo}: {URL.findall(texto)}")

# Fragmentos que la regex no encuentra
print("\nAlgunos fragmentos que la regex no encuentra:")
no_encontrados = muestra_desarrollo[~marcado & (muestra_desarrollo["contaminado"] == 1)]
for fragmento in no_encontrados["fragmento"].drop_duplicates().head(8):
    print("  ", fragmento)

regex: precisión 1.00, recall 0.40 (12 libros marcados)

Falsos positivos en desarrollo:
  ¿Que pasaría si…?: ['http://www.xkcd.com']

Algunos fragmentos que la regex no encuentra:
   El libro más esperado del año llega por fin en edición digital completa. Leé este y miles de libros más sin publicidad. Activá tu prueba gratuita desde tu celular.
   Conseguí este título y más de 50.000 libros en la app LetraTotal: el primer mes es sin cargo.
   Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.
   Entrá a librоexpress.invalid y leé sin límites.
   Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.
   Leé este y miles de libros más sin publicidad. Activá tu prueba gratuita desde tu celular.
   ¡Oferta por tiempo limitado! Suscribite hoy y llevate tres libros de regalo.
   Link en nuestro sitio: h t t p s : / / epubhub [.] example


**Qué muestra**

- En prueba, la regex tiene **precisión 1,00 y recall 0,40**: los 12 libros que marca están contaminados, pero encuentra solo 12 de los 30.
- **Detecta bien** los enlaces explícitos: con `http`, con `www.`, correos y teléfonos.
- **No detecta** los enlaces disimulados (letras separadas, letras cirílicas, «punto» en lugar de «.»), el texto promocional sin enlace («Ganá dinero desde tu casa recomendando libros»), los mensajes en otro idioma sin dirección ni los libros copiados.
- **Falso positivo en desarrollo:** *¿Qué pasaría si…?* menciona `http://www.xkcd.com`, el sitio real de su autor. La regex no distingue un sitio real de uno promocional.

## 2.2 Normalizar antes de buscar

Primero miramos qué caracteres raros hay en los fragmentos. `unicodedata` da el nombre y la categoría de cada carácter: la categoría `Cf` (*format*) agrupa los invisibles.

In [10]:
# Caracteres fuera del alfabeto latino (código mayor a 0x24F) o invisibles, en los fragmentos de desarrollo
raros = Counter(c for fragmento in muestra_desarrollo["fragmento"] for c in fragmento
                if ord(c) > 0x24F or unicodedata.category(c) == "Cf")
for c, n in raros.most_common():
    print(f"{c!r:10} {unicodedata.name(c, '?'):45} categoría {unicodedata.category(c)}  ({n} veces)")

# ¿Los invisibles aparecen también en sinopsis limpias?
tiene_invisibles = muestra_desarrollo["synopsis"].apply(lambda texto: any(unicodedata.category(c) == "Cf" for c in texto))
print("\nLibros con algún carácter invisible, según estén contaminados o no:")
print(pd.crosstab(tiene_invisibles, muestra_desarrollo["contaminado"]).to_string())

'\u200b'   ZERO WIDTH SPACE                              categoría Cf  (4 veces)
'\u2060'   WORD JOINER                                   categoría Cf  (3 veces)
'а'        CYRILLIC SMALL LETTER A                       categoría Ll  (2 veces)
'і'        CYRILLIC SMALL LETTER BYELORUSSIAN-UKRAINIAN I categoría Ll  (2 veces)
'е'        CYRILLIC SMALL LETTER IE                      categoría Ll  (2 veces)
'о'        CYRILLIC SMALL LETTER O                       categoría Ll  (1 veces)
'с'        CYRILLIC SMALL LETTER ES                      categoría Ll  (1 veces)

Libros con algún carácter invisible, según estén contaminados o no:


contaminado    0   1
synopsis            
False        276  63
True           4   7


In [11]:
# Letras cirílicas que se ven iguales a las latinas: а е о р с у х і → a e o p c y x i
HOMOGLIFOS = str.maketrans("аеорсухі", "aeopcyxi")


def normalizar(texto):
    """Deshace los disfraces de los enlaces para que la regex los encuentre."""
    # 1. Caracteres invisibles: se borran todos los de la categoría Cf
    texto = "".join(c for c in texto if unicodedata.category(c) != "Cf")
    # 2. Letras de otro alfabeto que se ven iguales a las latinas
    texto = texto.translate(HOMOGLIFOS)
    # 3. Palabras o marcas en lugar de signos: "tomoya punto test", "tomoya [.] test"
    texto = texto.replace("[.]", ".")
    texto = re.sub(r"\bpunto\b", ".", texto)
    texto = re.sub(r"\bbarra\b", "/", texto)
    texto = re.sub(r"\barroba\b", "@", texto)
    # 4. Letras sueltas separadas por espacios: "w w w" → "www", "h t t p s" → "https"
    texto = re.sub(r"(?<=\b\w) (?=\w\b)", "", texto)
    # 5. Sin espacios alrededor de los signos de una dirección: "libroplus . example" → "libroplus.example".
    #    También junta "fin. Luego", pero eso no crea falsos enlaces: la regex exige una terminación de dominio
    texto = re.sub(r"\s*([.:/@])\s*", r"\1", texto)
    return texto


def detectar_normalizacion(datos):
    return datos["synopsis"].apply(normalizar).str.contains(URL)


marcas_prueba["normalización"] = detectar_normalizacion(muestra_prueba)
medir("regex", marcas_prueba["regex"], muestra_prueba)
medir("regex + normalización", marcas_prueba["normalización"], muestra_prueba)

# Dos casos que la normalización recuperó: la regex no los encontraba en el texto original
recuperados = muestra_prueba[marcas_prueba["normalización"] & ~marcas_prueba["regex"]]
for fragmento in recuperados["fragmento"].head(2):
    print(f"\nOriginal:    {fragmento!r}")
    print(f"Normalizado: {normalizar(fragmento)!r}")

regex: precisión 1.00, recall 0.40 (12 libros marcados)


regex + normalización: precisión 1.00, recall 0.63 (19 libros marcados)

Original:    'Entrá a tоmoclub.example y leé sin límites.'
Normalizado: 'Entrá a tomoclub.example y leé sin límites.'

Original:    'Buscalo en w w w . novelalibre . example'
Normalizado: 'Buscalo en www.novelalibre.example'


**Qué muestra**

- Los fragmentos usan dos tipos de caracteres raros: **invisibles** de categoría Cf (el espacio de ancho cero y el *word joiner*) y **letras cirílicas** que se ven iguales a las latinas (а, е, о, с, і).
- **Los invisibles no alcanzan como evidencia:** 4 de los 280 libros limpios de desarrollo también tienen alguno (dos con guion de separación silábica y dos con *word joiner*). Por eso no marcamos un libro por tenerlos: los borramos y volvemos a buscar con la regex.
- Con la normalización, el recall sube de **0,40 a 0,63** sin perder precisión (1,00).
- **Dos casos recuperados:** «tоmoclub.example» tenía una «о» cirílica, que la regex no reconoce como letra latina, y «w w w . novelalibre . example» tenía las letras y los puntos separados por espacios.
- El 37 % que sigue sin detectar no tiene ninguna dirección: es texto promocional o mensajes en otro idioma.

## 2.3 Detectar el idioma de cada oración

Primero, el idioma de la sinopsis completa con `langdetect`.

In [12]:
idioma_sinopsis = muestra_desarrollo["synopsis"].apply(detect)
print(pd.crosstab(idioma_sinopsis, muestra_desarrollo["contaminado"]).to_string())

contaminado    0   1
synopsis            
es           280  70


Ahora oración por oración, con `detect_langs`, que devuelve la probabilidad de cada idioma. Para cada combinación de cantidad mínima de palabras y umbral de probabilidad, una sinopsis queda marcada si alguna oración no está en español.

In [13]:
def idioma_mas_probable(oracion):
    """El idioma más probable de la oración y su probabilidad (detect_langs los ordena de mayor a menor)."""
    try:
        mejor = detect_langs(oracion)[0]
        return mejor.lang, mejor.prob
    except LangDetectException:   # oraciones sin letras, como una lista de números
        return "?", 0.0


oraciones_desarrollo = tabla_oraciones(muestra_desarrollo)
oraciones_desarrollo["palabras"] = oraciones_desarrollo["oracion"].str.split().str.len()
# zip(*...) separa la lista de pares (idioma, probabilidad) en dos columnas
oraciones_desarrollo["idioma"], oraciones_desarrollo["prob"] = zip(*oraciones_desarrollo["oracion"].apply(idioma_mas_probable))

filas = []
for minimo in [3, 5, 8, 10, 12]:
    for umbral in [0.5, 0.9, 0.99]:
        en_otro_idioma = oraciones_desarrollo[(oraciones_desarrollo["idioma"] != "es")
                                              & (oraciones_desarrollo["palabras"] >= minimo)
                                              & (oraciones_desarrollo["prob"] >= umbral)]
        marcado = muestra_desarrollo["slug"].isin(en_otro_idioma["slug"])
        filas.append({"mínimo de palabras": minimo, "umbral": umbral,
                      "precisión": precision_score(muestra_desarrollo["contaminado"], marcado, zero_division=0),
                      "recall": recall_score(muestra_desarrollo["contaminado"], marcado),
                      "marcados": marcado.sum()})
pd.DataFrame(filas).round(2)

,mínimo de palabras,umbral,precisión,recall,marcados
0,3,0.50,0.46,0.24,37
1,3,0.90,0.56,0.21,27
2,3,0.99,0.56,0.21,27
3,5,0.50,0.58,0.21,26
4,5,0.90,0.61,0.20,23
5,5,0.99,0.61,0.20,23
6,8,0.50,0.67,0.14,15
7,8,0.90,0.80,0.11,10
8,8,0.99,0.80,0.11,10
9,10,0.50,0.71,0.14,14


In [14]:
# Las oraciones "no españolas" con pocas palabras: casi todas son falsos positivos
cortas = oraciones_desarrollo[(oraciones_desarrollo["idioma"] != "es") & (oraciones_desarrollo["palabras"] < 10)]
cortas[["oracion", "palabras", "idioma", "prob"]].head(10).round(2)

,oracion,palabras,idioma,prob
50,Activá tu prueba gratuita desde tu celular.,7,ro,1.00
74,¿Por qué existe este caos?,5,pt,1.00
93,ELIGE AHORA… ¿SERÁS EL MAGO O EL GUERRERO?,8,pt,0.86
97,¿LA MAGIA O LA ESPADA?,5,pt,1.00
409,Activá tu prueba gratuita desde tu celular.,7,ro,1.00
706,¿Qué hubo antes del Big Bang?,6,tl,0.86
779,¿Patines cohete?,2,en,0.86
780,¿Qué invento era ése?,4,ca,0.57
805,¿O tal vez si los tengan?,6,hu,0.71
819,"David Yagüe, 20 Minutos «Un narrador excepcional».",7,pt,1.00


**Qué muestra**

- **Con la sinopsis completa, langdetect dice «español» para las 350**, también para las 70 contaminadas: el mensaje insertado es una o dos oraciones dentro de un texto largo en español, y no alcanza para cambiar el resultado.
- **Por oración, las cortas generan falsos positivos.** Con 3 palabras como mínimo se marcan 37 libros con precisión 0,46. Los errores son preguntas cortas, oraciones en mayúsculas y listas de nombres: «¿LA MAGIA O LA ESPADA?» sale como portugués.
- **Elegimos 10 palabras y umbral 0,9:** es la combinación con mayor precisión en desarrollo (0,89), con el mismo recall que con 8 palabras (0,11). Con 12 palabras se pierden mensajes reales y el recall baja a 0,04.
- **El umbral importa menos que el largo:** entre 0,9 y 0,99 no cambia nada, porque las probabilidades de `detect_langs` en oraciones largas están casi siempre cerca de 1.

In [15]:
MIN_PALABRAS = 10
UMBRAL_IDIOMA = 0.9


def detectar_idioma(datos):
    """Marca las sinopsis con alguna oración de MIN_PALABRAS o más que no esté en español."""
    orac = tabla_oraciones(datos)
    orac = orac[orac["oracion"].str.split().str.len() >= MIN_PALABRAS].copy()   # las cortas ni se analizan
    orac["idioma"], orac["prob"] = zip(*orac["oracion"].apply(idioma_mas_probable))
    en_otro_idioma = orac[(orac["idioma"] != "es") & (orac["prob"] >= UMBRAL_IDIOMA)]
    return datos["slug"].isin(en_otro_idioma["slug"])


marcas_prueba["idioma"] = detectar_idioma(muestra_prueba)
medir("idioma por oración", marcas_prueba["idioma"], muestra_prueba)

idioma por oración: precisión 0.89, recall 0.27 (9 libros marcados)


**Qué muestra**

- En prueba: **precisión 0,89 y recall 0,27.** Marca 9 libros: 8 tienen mensajes en inglés o portugués («Download this ebook for FREE…», «Leia milhares de livros…») y 1 es un falso positivo, *El rey de amarillo*, que cita títulos en inglés.
- El recall es bajo porque la mayor parte de la contaminación está en español: este detector solo puede encontrar la que está en otro idioma.

## 2.4 Reconocer entidades

Aplicamos NER con spaCy (`es_core_news_lg`) a las oraciones de los fragmentos de desarrollo y a la misma cantidad de oraciones de sinopsis limpias, elegidas al azar.

In [16]:
nlp = spacy.load("es_core_news_lg")

oraciones_fragmento = [o for fragmento in muestra_desarrollo.loc[muestra_desarrollo["contaminado"] == 1, "fragmento"]
                       for o in oraciones(fragmento)]
oraciones_limpias = tabla_oraciones(muestra_desarrollo[muestra_desarrollo["contaminado"] == 0])["oracion"].sample(
    n=len(oraciones_fragmento), random_state=SEMILLA)


def contar_entidades(textos):
    """Cuenta las entidades por etiqueta y por texto. nlp.pipe procesa muchos textos de una vez."""
    por_etiqueta, por_texto = Counter(), Counter()
    for doc in nlp.pipe(textos):
        for ent in doc.ents:
            por_etiqueta[ent.label_] += 1
            por_texto[f"{ent.text} ({ent.label_})"] += 1
    return por_etiqueta, por_texto


etiquetas_fragmento, textos_fragmento = contar_entidades(oraciones_fragmento)
etiquetas_limpias, textos_limpios = contar_entidades(oraciones_limpias)
print(f"{len(oraciones_fragmento)} oraciones de cada grupo\n")
print(pd.DataFrame({"fragmentos": etiquetas_fragmento, "limpias": etiquetas_limpias}).fillna(0).astype(int).to_string())
print("\nEntidades más frecuentes en los fragmentos:", textos_fragmento.most_common(12))
print("\nEntidades más frecuentes en las oraciones limpias:", textos_limpios.most_common(12))

88 oraciones de cada grupo

      fragmentos  limpias
MISC          37       32
LOC           15       27
PER           16       49
ORG            2        5

Entidades más frecuentes en los fragmentos: [('compartila (PER)', 7), ('PDF (MISC)', 6), ('Ganá (PER)', 5), ('Download this ebook for FREE (MISC)', 4), ('sign-up required (MISC)', 4), ('EPUB sin registro (MISC)', 4), ('Descargalo (MISC)', 4), ('Oferta (LOC)', 2), ('EPUB (MISC)', 2), ('LibroLibre (MISC)', 2), ('EpubMax (MISC)', 2), ('Visitá pagin\u2060azona.invalid (LOC)', 1)]

Entidades más frecuentes en las oraciones limpias: [('Aprender (MISC)', 3), ('Lavoisier (PER)', 2), ('Haruf (LOC)', 2), ('Díaz (PER)', 2), ('Tom (PER)', 1), ('Peter (PER)', 1), ('Andes (LOC)', 1), ('Tarl Cabot (PER)', 1), ('Gor (LOC)', 1), ('Contratierra (LOC)', 1), ('¿Cómo (MISC)', 1), ('Wallace Price (PER)', 1)]


**Qué muestra**

- El modelo de spaCy en español tiene cuatro etiquetas: PER, LOC, ORG y MISC. **En las oraciones limpias predominan las personas (49) y los lugares (27):** autores, personajes y ciudades.
- **En los fragmentos predomina MISC (37 de 70 entidades):** formatos (*PDF*, *EPUB*), marcas inventadas (*LibroLibre*, *EpubMax*) y frases enteras en inglés (*Download this ebook for FREE*). Las «personas» que encuentra son errores: «Ganá» y «compartila» son verbos con mayúscula al comienzo de la oración.
- **¿Qué entidades aparecen en el texto insertado y no en una sinopsis normal?** Formatos de archivo, nombres de marcas o apps y texto en otro idioma etiquetado como MISC.
- **¿Sirven para detectar o solo para explicar?** Para explicar. MISC también es frecuente en las oraciones limpias (32), y con el spam el modelo se equivoca mucho, así que una regla del tipo «tiene MISC» marcaría muchos libros legítimos. En un libro que ya marcó otro detector, en cambio, las entidades muestran qué marca o qué formato se promociona.

## 2.5 Clasificar oraciones

Armamos un conjunto de oraciones con los libros de desarrollo: las oraciones del fragmento son positivas y las de las sinopsis limpias, negativas. El resto del texto de los libros contaminados (la sinopsis original) no se usa. Entrenamos dos clasificadores, uno con TF-IDF y otro con embeddings, los dos con `LogisticRegression(class_weight="balanced", max_iter=1000)`.

Para los embeddings usamos `multilingual-e5-small`, uno de los dos modelos de oración del TP2: es multilingüe y su límite de 512 tokens le permite ver las sinopsis completas, que necesitamos en los pasos 5 a 8.

In [17]:
filas = []
for slug, sinopsis, contaminado, fragmento in zip(muestra_desarrollo["slug"], muestra_desarrollo["synopsis"],
                                                  muestra_desarrollo["contaminado"], muestra_desarrollo["fragmento"]):
    if contaminado == 1:
        filas += [(slug, o, 1) for o in oraciones(fragmento)]
    else:
        filas += [(slug, o, 0) for o in oraciones(sinopsis)]
oraciones_entrenamiento = pd.DataFrame(filas, columns=["slug", "oracion", "spam"])
print(f"{len(oraciones_entrenamiento)} oraciones: {oraciones_entrenamiento['spam'].sum()} positivas "
      f"({oraciones_entrenamiento.loc[oraciones_entrenamiento['spam'] == 1, 'oracion'].nunique()} distintas)")

e5 = SentenceTransformer("intfloat/multilingual-e5-small")


def codificar(textos):
    """Embeddings de E5, normalizados. E5 pide un prefijo según el papel del texto: en el TP2 usamos
    "passage: " para los libros, que se buscaban con consultas. Acá los textos son la entrada de un
    clasificador o se comparan entre sí, y para eso el modelo recomienda "query: "."""
    return e5.encode(["query: " + texto for texto in textos], normalize_embeddings=True, batch_size=64)


def codificar_con_cache(textos, ruta):
    """Como codificar, pero guarda el resultado en disco para no recalcularlo (tarda varios minutos).
    Si cambia el texto, hay que borrar el archivo .npy para que se vuelva a calcular."""
    if os.path.exists(ruta):
        E = np.load(ruta)
    else:
        E = codificar(textos)
        np.save(ruta, E)
    assert len(E) == len(textos), f"{ruta} no corresponde a estos textos: borrarlo y volver a ejecutar"
    return E


E_oraciones_entrenamiento = codificar(oraciones_entrenamiento["oracion"].tolist())

1831 oraciones: 88 positivas (62 distintas)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2997.52it/s]

**Cómo elegimos el umbral.** Una sinopsis es sospechosa si alguna de sus oraciones supera el umbral, así que el puntaje de un libro es la probabilidad de su oración más sospechosa. El umbral sale de `precision_recall_curve` sobre los libros de desarrollo: el más bajo que alcanza la precisión buscada, que es el de mayor recall entre los que la alcanzan.

**Qué precisión buscamos.** El enunciado propone, por ejemplo, 0,90. Pedimos 0,97 porque la muestra está enriquecida: el 20 % de sus libros está contaminado, y en el catálogo la proporción es bastante menor. Con la misma regla, cada libro contaminado que se encuentra en el catálogo viene acompañado de más libros limpios marcados por error que en la muestra, así que una precisión de 0,90 en la muestra sería bastante menor en el catálogo.

Si calculáramos la curva con las mismas oraciones con que entrenamos, el modelo ya las habría visto y las probabilidades saldrían optimistas. Por eso usamos `cross_val_predict`: divide las oraciones en 5 partes, entrena con 4 y predice la quinta, así cada oración recibe la probabilidad de un modelo que no la vio. `GroupKFold` deja todas las oraciones de un libro en la misma parte, porque la guía pide dividir por libro.

In [18]:
def puntaje_por_libro(orac, probabilidades, datos):
    """Probabilidad de cada libro de datos: la de su oración más sospechosa (0 si no tiene oraciones)."""
    por_libro = pd.Series(probabilidades).groupby(orac["slug"].to_numpy()).max()
    return por_libro.reindex(datos["slug"], fill_value=0).to_numpy()


PRECISION_BUSCADA = 0.97


def umbral_por_precision(y, puntajes, precision_buscada):
    """El umbral más bajo cuya precisión alcanza la buscada, con su precisión y su recall."""
    precision, recall, umbrales = precision_recall_curve(y, puntajes)
    # precision y recall tienen un valor más que umbrales (el último punto no tiene umbral): se descarta
    alcanza = precision[:-1] >= precision_buscada
    assert alcanza.any(), "ningún umbral alcanza esa precisión"
    i = np.argmax(alcanza)   # argmax de un arreglo de True/False da la posición del primer True
    return umbrales[i], precision[i], recall[i]


y_oraciones = oraciones_entrenamiento["spam"]
grupos = GroupKFold(n_splits=5)
clasificadores = {
    # Con TF-IDF se usa un pipeline, para que el vectorizador se ajuste dentro de cada partición
    "TF-IDF": (make_pipeline(TfidfVectorizer(), LogisticRegression(class_weight="balanced", max_iter=1000)),
               oraciones_entrenamiento["oracion"]),
    "embeddings": (LogisticRegression(class_weight="balanced", max_iter=1000), E_oraciones_entrenamiento),
}
umbrales = {}
for nombre, (modelo, X) in clasificadores.items():
    # predict_proba devuelve una columna por clase, en el orden de classes_ ([0, 1]): la columna 1 es spam
    fuera_de_muestra = cross_val_predict(modelo, X, y_oraciones, groups=oraciones_entrenamiento["slug"],
                                         cv=grupos, method="predict_proba")[:, 1]
    puntajes = puntaje_por_libro(oraciones_entrenamiento, fuera_de_muestra, muestra_desarrollo)
    umbrales[nombre], p, r = umbral_por_precision(muestra_desarrollo["contaminado"], puntajes, PRECISION_BUSCADA)
    marcados_05 = (puntajes >= 0.5).sum()
    print(f"{nombre}: umbral {umbrales[nombre]:.3f} (en desarrollo: precisión {p:.2f}, recall {r:.2f}). "
          f"Con 0,5 marcaría {marcados_05} libros")
    modelo.fit(X, y_oraciones)   # el modelo final se entrena con todas las oraciones de desarrollo

TF-IDF: umbral 0.534 (en desarrollo: precisión 0.97, recall 0.93). Con 0,5 marcaría 67 libros
embeddings: umbral 0.766 (en desarrollo: precisión 0.98, recall 0.84). Con 0,5 marcaría 96 libros


In [19]:
def probabilidades_spam(orac, nombre, embeddings=None):
    """Probabilidad de spam de cada oración según el clasificador. Para el de embeddings se pueden
    pasar los embeddings de las oraciones ya calculados."""
    if nombre == "TF-IDF":
        X = orac["oracion"]
    else:
        X = embeddings if embeddings is not None else codificar(orac["oracion"].tolist())
    return clasificadores[nombre][0].predict_proba(X)[:, 1]


def detectar_clasificador(datos, nombre, embeddings=None):
    """Marca las sinopsis con alguna oración cuya probabilidad de spam alcance el umbral del clasificador."""
    orac = tabla_oraciones(datos)
    puntajes = puntaje_por_libro(orac, probabilidades_spam(orac, nombre, embeddings), datos)
    return pd.Series(puntajes >= umbrales[nombre], index=datos.index)


for nombre in clasificadores:
    marcas_prueba[f"clasificador {nombre}"] = detectar_clasificador(muestra_prueba, nombre)
    medir(f"clasificador {nombre}", marcas_prueba[f"clasificador {nombre}"], muestra_prueba)

clasificador TF-IDF: precisión 1.00, recall 0.90 (27 libros marcados)


clasificador embeddings: precisión 1.00, recall 0.63 (19 libros marcados)


In [20]:
# ¿Qué contaminados de prueba encuentra cada uno?
contaminados = muestra_prueba["contaminado"] == 1
detalle = pd.DataFrame({"fragmento": muestra_prueba["fragmento"].str[:90],
                        "TF-IDF": marcas_prueba["clasificador TF-IDF"].map({True: "✓", False: "·"}),
                        "embeddings": marcas_prueba["clasificador embeddings"].map({True: "✓", False: "·"})})[contaminados]
detalle.sort_values(["TF-IDF", "embeddings"]).head(12)

,fragmento,TF-IDF,embeddings
393,Pedí tu copia escribiendo a ventas@tomozona.test,·,·
285,Entrá a letrаya.invalid y leé sin límites.,·,·
325,Link en nuestro sitio: h t t p s : / / lectortotal [.] example,·,·
65,Una obra imprescindible que no puede faltar en tu biblioteca personal. Más títulos como es,✓,·
322,Entrá a tоmoclub.example y leé sin límites.,✓,·
194,Buscalo en w w w . novelalibre . example,✓,·
98,"Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector ele",✓,·
430,Buscalo en w w w . letramax . example,✓,·
27,Descargalo gratis en TomoMax punto example barra cardamomo,✓,·
268,Pedí tu copia escribiendo a ventas@letrazona.invalid,✓,·


In [21]:
# Frases nuevas, que no están en la muestra: tres promocionales con el mismo sentido que las vistas y una legítima
nuevas = ["Bajate la novela completa sin pagar nada desde nuestra página.",
          "Sumate a nuestro club de lectura y recibí un libro de regalo cada mes.",
          "Contactanos por mensaje privado para conseguir tu ejemplar digital.",
          "La novela narra la vida de un joven pescador en un pueblo de la costa gallega."]
probabilidades = pd.DataFrame({
    "TF-IDF": clasificadores["TF-IDF"][0].predict_proba(nuevas)[:, 1],
    "embeddings": clasificadores["embeddings"][0].predict_proba(codificar(nuevas))[:, 1]}, index=nuevas)
print("Umbrales:", {nombre: round(u, 3) for nombre, u in umbrales.items()})
probabilidades.round(3)

Umbrales: {'TF-IDF': np.float64(0.534), 'embeddings': np.float32(0.766)}


,TF-IDF,embeddings
Bajate la novela completa sin pagar nada desde nuestra página.,0.787,0.914
Sumate a nuestro club de lectura y recibí un libro de regalo cada mes.,0.425,0.809
Contactanos por mensaje privado para conseguir tu ejemplar digital.,0.509,0.843
La novela narra la vida de un joven pescador en un pueblo de la costa gallega.,0.022,0.199


**Qué muestra**

- **A igual precisión en desarrollo, TF-IDF encuentra más contaminados de prueba:** recall 0,90 contra 0,63, los dos sin falsos positivos.
- **Con el mismo umbral, los modelos no son comparables:** con 0,5, TF-IDF marcaría 67 libros de desarrollo y el de embeddings, 96. Por eso cada uno tiene su umbral (0,534 y 0,766).
- **Dónde gana cada uno en prueba:** TF-IDF reconoce cadenas literales que se repiten en los fragmentos («ventas@», «punto example barra», «Si te gustó esta reseña»). El de embeddings encuentra los mensajes de sentido claramente promocional («Descargá el libro completo gratis…»), pero no las direcciones disimuladas: para él, «Entrá a tоmoclub.example y leé sin límites» es una oración común.
- **Con frases nuevas generaliza mejor el de embeddings.** Las tres frases promocionales inventadas superan su umbral (de 0,81 a 0,91). TF-IDF solo marca la primera (0,79), que comparte palabras con los fragmentos («completa», «sin pagar nada»). Ninguno marca la frase legítima.
- **¿Por qué clasificar oraciones y no la sinopsis completa?** El mensaje insertado es una o dos oraciones dentro de un texto largo y legítimo. En la sinopsis completa su señal se diluye, como le pasó a langdetect con el idioma. Clasificando oraciones, alcanza con que una sola sea promocional, y los ejemplos positivos son solo el texto insertado.
- **¿Qué representación generaliza mejor?** Los embeddings, para mensajes nuevos con el mismo sentido. TF-IDF detecta mejor las cadenas literales, como una URL o un nombre de dominio. En el catálogo se confirma: solo el de embeddings encuentra mensajes que no están en la muestra (ver 2.7).

## 2.6 Encontrar libros copiados

Algunos libros no existen: copian la sinopsis de otro libro del catálogo y le agregan texto. Para cada libro del catálogo buscamos su vecino más parecido con `NearestNeighbors` (similitud del coseno sobre TF-IDF) y revisamos los pares con similitud alta y **ningún autor en común**.

- **¿Cuál es la copia?** La de sinopsis más larga: tiene el texto del original más lo que le agregaron.
- **Confirmación con Jaccard.** Una copia contiene casi todas las palabras del original, así que su Jaccard sobre conjuntos de palabras es alto. Las series legítimas de distintos autores (por ejemplo, las colecciones juveniles) comparten un texto promocional, pero cada libro tiene además su propia trama.

In [22]:
X_catalogo = TfidfVectorizer().fit_transform(libros["synopsis"])
vecinos = NearestNeighbors(n_neighbors=2, metric="cosine").fit(X_catalogo)
distancias, indices = vecinos.kneighbors(X_catalogo)
# Cada libro sale como su propio vecino más cercano (distancia 0); el otro vecino es el más parecido.
# Se compara con la posición del libro, por si dos sinopsis empatan en distancia 0
propio = indices[:, 0] == np.arange(len(libros))
libros["vecino"] = np.where(propio, indices[:, 1], indices[:, 0])
libros["similitud_vecino"] = 1 - np.where(propio, distancias[:, 1], distancias[:, 0])   # distancia coseno = 1 - similitud


def palabras(texto):
    return set(re.findall(r"\w+", texto.lower()))


def jaccard(a, b):
    """Palabras en común sobre palabras en total."""
    return len(a & b) / len(a | b)


filas = []
for i, j in enumerate(libros["vecino"]):
    sin_autor_comun = not set(libros.at[i, "authors"]) & set(libros.at[j, "authors"])
    # i < j: cada par una sola vez (si los dos son vecinos mutuos, aparecería dos veces)
    if libros.at[i, "similitud_vecino"] >= 0.75 and sin_autor_comun and (i < j or libros.at[j, "vecino"] != i):
        # La copia es la de sinopsis más larga
        copia, original = (i, j) if len(libros.at[i, "synopsis"]) > len(libros.at[j, "synopsis"]) else (j, i)
        filas.append({"copia": libros.at[copia, "slug"], "autor_copia": ", ".join(libros.at[copia, "authors"]),
                      "original": libros.at[original, "slug"], "autor_original": ", ".join(libros.at[original, "authors"]),
                      "coseno": round(libros.at[i, "similitud_vecino"], 2),
                      "jaccard": round(jaccard(palabras(libros.at[copia, "synopsis"]), palabras(libros.at[original, "synopsis"])), 2)})
pares_copia = pd.DataFrame(filas).sort_values("jaccard")
pares_copia["es_copia"] = pares_copia["jaccard"] >= 0.7
copias = set(pares_copia.loc[pares_copia["es_copia"], "copia"])
print(f"Pares con coseno ≥ 0,75 y autores distintos: {len(pares_copia)}; confirmados como copia: {len(copias)}")
pares_copia.head(14)

Pares con coseno ≥ 0,75 y autores distintos: 72; confirmados como copia: 66


,copia,autor_copia,original,autor_original,coseno,jaccard,es_copia
31,nivel-26,Anthony E. Zuiker,no-podras-esconderte,"Anthony Zuiker, Duane Swierczynski",0.76,0.53,False
21,el-desafio-del-caballero-lobo,R. L. Stine,quien-secuestro-a-la-princesa-saralinda,"Bill Stine, Megan Stine",0.79,0.57,False
20,la-amenaza-del-dueno-del-tiempo,Lynn Beach,quien-secuestro-a-la-princesa-saralinda,"Bill Stine, Megan Stine",0.79,0.60,False
22,los-caballeros-fantasmas-de-camelot,David Anthony Kraft,la-venganza-de-la-reina-de-dragones,Eric Affabee,0.77,0.63,False
57,la-jornada-de-un-periodista-americano-en-2889-edicion-shjv,Michel Verne,la-jornada-de-un-periodista-americano-en-2890-edicion-shjv,Julio Verne,0.79,0.65,False
23,quien-secuestro-a-la-princesa-saralinda,"Bill Stine, Megan Stine",la-venganza-de-la-reina-de-dragones,Eric Affabee,0.80,0.66,False
16,sombras-sobre-el-rio-2,Andrés Barrionuevo,peter-pan,J. M. Barrie,0.78,0.71,True
47,los-hijos-de-la-llanura,Lucía Correa,la-fascinante-historia-de-las-palabras,Ricardo Soca,0.81,0.72,True
59,sombras-sobre-el-relojero,Julián Ledesma,misterio-del-fugitivo,Enid Blyton,0.87,0.72,True
45,memorias-del-bosque-2,Clara Arrieta,como-crear-una-mente,Ray Kurzweil,0.86,0.74,True


In [23]:
def detectar_copia(datos):
    return datos["slug"].isin(copias)


marcas_prueba["copia"] = detectar_copia(muestra_prueba)
medir("libros copiados", marcas_prueba["copia"], muestra_prueba)

libros copiados: precisión 1.00, recall 0.10 (3 libros marcados)


**Qué muestra**

- **72 pares** tienen coseno ≥ 0,75 y ningún autor en común; **66 se confirman como copia** con Jaccard ≥ 0,7.
- **Los 6 rechazados son libros legítimos.** Cuatro son de una misma colección juvenil con autores distintos (*El desafío del caballero Lobo*, *La amenaza del dueño del tiempo*…): comparten la presentación de la colección, pero cada uno tiene su propia trama, y por eso su Jaccard es más bajo (0,53 a 0,66). Los otros dos son el mismo autor escrito de dos formas («Anthony E. Zuiker» y «Anthony Zuiker») y *La jornada de un periodista americano*, atribuida a Michel Verne en una edición y a Julio Verne en otra.
- **Las copias tienen títulos y autores inventados:** «Sombras sobre el bosque», de «Paula Oyarzún», copia la sinopsis de *Bodas de sangre*, de García Lorca, y le agrega texto. En todas, la copia es la sinopsis más larga del par.
- **Errores:** dos libros marcados solo por este detector son legítimos: *Richard P. Feynman. La física de las palabras* y *Las grandes preguntas de los niños* tienen casi la misma sinopsis que otro libro, atribuido a otro autor.
- En prueba: **precisión 1,00 y recall 0,10** (3 de los 30). Es un tipo de contaminación poco frecuente en la muestra.

## 2.7 Combinar y entregar

Un libro es sospechoso si al menos uno de los detectores de la combinación lo marca. La combinación usa la regex sobre el texto normalizado (que incluye lo que encuentra la regex sola), los dos clasificadores y los libros copiados. El detector de idioma queda afuera: en la parte de prueba no encuentra ningún contaminado que no encuentren los demás, y más abajo se ve que lo que marca por su cuenta en el catálogo son falsos positivos.

In [24]:
COMBINACION = ["normalización", "clasificador TF-IDF", "clasificador embeddings", "copia"]
# Un libro queda marcado por la combinación si alguno de los detectores elegidos lo marca
marcas_prueba["combinación"] = pd.concat([marcas_prueba[d] for d in COMBINACION], axis=1).any(axis=1)

solo_idioma = marcas_prueba["idioma"] & ~marcas_prueba["combinación"]
print(f"Libros de prueba que marca el detector de idioma y no la combinación: {solo_idioma.sum()} "
      f"(contaminados: {muestra_prueba.loc[solo_idioma, 'contaminado'].sum()})")

y = muestra_prueba["contaminado"]
tabla = pd.DataFrame({detector: {"precisión": precision_score(y, marcado, zero_division=0),
                                 "recall": recall_score(y, marcado),
                                 "marcados": marcado.sum(),
                                 "falsos positivos": (marcado & (y == 0)).sum()}
                      for detector, marcado in marcas_prueba.items()}).T
tabla.astype({"marcados": int, "falsos positivos": int}).round(2)

Libros de prueba que marca el detector de idioma y no la combinación: 1 (contaminados: 0)


,precisión,recall,marcados,falsos positivos
regex,1.00,0.40,12,0
normalización,1.00,0.63,19,0
idioma,0.89,0.27,9,1
clasificador TF-IDF,1.00,0.90,27,0
clasificador embeddings,1.00,0.63,19,0
copia,1.00,0.10,3,0
combinación,1.00,1.00,30,0


**Qué muestra**

- **La combinación marca los 30 contaminados de prueba sin falsos positivos** (precisión y recall 1,00). Ningún detector solo lo logra: el de mayor recall, el clasificador TF-IDF, encuentra 27.
- **Este resultado es optimista.** La muestra está enriquecida y sus mensajes son los mismos con los que desarrollamos los detectores. En el catálogo la proporción de contaminados es menor y hay mensajes nuevos, así que esperamos menos precisión y menos recall.
- El detector de idioma tiene la peor precisión (0,89) y lo único que agregaría a la combinación es su falso positivo.

Ahora aplicamos los detectores a todo el catálogo que quedó del paso 1. Las oraciones del catálogo son más de 50 mil: sus embeddings se calculan una vez (unos 10 minutos en CPU) y quedan guardados en `data/`.

In [25]:
oraciones_catalogo = tabla_oraciones(libros)
E_oraciones_catalogo = codificar_con_cache(oraciones_catalogo["oracion"].tolist(), "data/embeddings_oraciones_catalogo.npy")

marcas_catalogo = pd.DataFrame({
    "regex": detectar_regex(libros),
    "normalizacion": detectar_normalizacion(libros),
    "idioma": detectar_idioma(libros),
    "clasificador_tfidf": detectar_clasificador(libros, "TF-IDF"),
    "clasificador_embeddings": detectar_clasificador(libros, "embeddings", E_oraciones_catalogo),
    "copia": detectar_copia(libros)})
# Los mismos detectores que la combinación de la tabla de prueba (normalizacion incluye a regex)
COMBINACION_CATALOGO = ["normalizacion", "clasificador_tfidf", "clasificador_embeddings", "copia"]
sospechoso = marcas_catalogo[COMBINACION_CATALOGO].any(axis=1)
print(f"Libros marcados por cada detector:\n{marcas_catalogo.sum().to_string()}")
print(f"\nSospechosos: {sospechoso.sum()} de {len(libros)} ({sospechoso.mean():.1%})")

# Lo que marca el detector de idioma por su cuenta: la oración que lo dispara en algunos de esos libros
solo_idioma = marcas_catalogo["idioma"] & ~sospechoso
print(f"\nLibros que marca solo el detector de idioma: {solo_idioma.sum()}. Algunas de sus oraciones:")
for texto in libros.loc[solo_idioma, "synopsis"].head(8):
    for oracion in oraciones(texto):
        idioma, prob = idioma_mas_probable(oracion)
        if len(oracion.split()) >= MIN_PALABRAS and idioma != "es" and prob >= UMBRAL_IDIOMA:
            print(f"  [{idioma}] {oracion[:130]}")
            break

Libros marcados por cada detector:
regex                      128
normalizacion              177
idioma                     120
clasificador_tfidf         265
clasificador_embeddings    282
copia                       66

Sospechosos: 406 de 8625 (4.7%)

Libros que marca solo el detector de idioma: 64. Algunas de sus oraciones:
  [en] Al Pacino, Harrison Ford, Philip Seymour Hoffman, Abel Ferrara, Tom Hanks, Bill Murray, Scarlett Johansson, Sofia Coppola, Nicole 
  [en] Fotografías de: Mo Anthoine; John Cleare; Hamish MacInnes y Chris Mikami.
  [pt] No importa si prefieres tomar notas de forma física o digital.
  [ca] Él le hace recordar al primer hombre que ella amó.
  [pt] ¡UN HORRIBLE HOMBRE LOBO AMENAZA EL REINO, EMBUTIDO EN UNA ARMADURA DE CABALLERO!
  [en] CRIATURAS SINIESTRAS CUSTODIAN LOS TESOROS DEL CASTILLO DEL CUERVO MALDITO.
  [de] ¡LOS ESPÍRITUS DE LOS LEGENDARIOS CABALLEROS DE CAMELOT HAN REGRESADO CON INTENCIONES ASESINAS!
  [de] SE PRECISAN DOS DADOS DE 10 CARAS Y UNO D

In [26]:
# El motivo junta los nombres de los detectores que marcaron el libro, por ejemplo "regex+normalizacion"
MOTIVOS = ["regex"] + COMBINACION_CATALOGO
motivos = marcas_catalogo[MOTIVOS].apply(lambda fila: "+".join(nombre for nombre, marcado in fila.items() if marcado), axis=1)
sospechosos = pd.DataFrame({"slug": libros["slug"], "motivo": motivos})[sospechoso]
sospechosos.to_csv(RUTA_SOSPECHOSOS, index=False)

limpio = libros.loc[~sospechoso, ["slug", "title", "authors", "synopsis"]].reset_index(drop=True)
limpio["authors"] = limpio["authors"].str.join(", ")   # en el CSV, los autores como texto: "Autor 1, Autor 2"
limpio.to_csv(RUTA_LIMPIO, index=False)
print(f"{RUTA_SOSPECHOSOS}: {len(sospechosos)} libros. {RUTA_LIMPIO}: {len(limpio)} libros")
print("\nMotivos más frecuentes:")
print(sospechosos["motivo"].value_counts().head(8).to_string())

data/grupoNN_sospechosos.csv: 406 libros. data/catalogo_limpio.csv: 8219 libros

Motivos más frecuentes:
motivo
clasificador_embeddings                                                 84
clasificador_tfidf+clasificador_embeddings                              79
regex+normalizacion+clasificador_tfidf+clasificador_embeddings          53
clasificador_tfidf                                                      38
regex+normalizacion                                                     32
clasificador_tfidf+clasificador_embeddings+copia                        23
regex+normalizacion+clasificador_tfidf+clasificador_embeddings+copia    21
normalizacion+clasificador_tfidf                                        15


In [27]:
# Los casos más dudosos son los que marcó un solo clasificador.
# Para algunos de esos libros, su oración más sospechosa y la probabilidad que le da el clasificador
cantidad = marcas_catalogo[COMBINACION_CATALOGO].sum(axis=1)   # cuántos detectores de la combinación lo marcan
for columna, nombre in [("clasificador_tfidf", "TF-IDF"), ("clasificador_embeddings", "embeddings")]:
    solo = libros.loc[marcas_catalogo[columna] & (cantidad == 1), "slug"]
    orac = oraciones_catalogo[oraciones_catalogo["slug"].isin(solo)].copy()
    orac["prob"] = probabilidades_spam(orac, nombre, E_oraciones_catalogo[orac.index])
    # La oración más sospechosa de cada libro: se ordena por probabilidad y se deja la primera de cada slug
    mas_sospechosas = orac.sort_values("prob", ascending=False).drop_duplicates("slug")
    print(f"\nSolo el clasificador {nombre}: {len(solo)} libros. Algunas de sus oraciones más sospechosas:")
    for _, fila in mas_sospechosas.sample(8, random_state=SEMILLA).iterrows():
        print(f"  {fila['prob']:.2f}  {fila['oracion'][:120]}")


Solo el clasificador TF-IDF: 38 libros. Algunas de sus oraciones más sospechosas:
  0.54  ¡No te engañes!
  0.54  PON EN JUEGO TU INTELIGENCIA DOMÉSTICA.
  0.77  Inspirado por la obra de Cervantes, Sam DuChamp, un escritor mediocre de Si te gustó esta reseña, compartila con tus ami
  0.60  Sin fronteras, sin límites.
  0.54  Si rompes y retomas tu relación continuamente.
  0.56  Te ofrecemos las claves para : • preparar tu incorporación a tu puesto de trabajo y no caer en los errores más comunes; 
  0.68  ¡Tu bebé te lo agradecerá!
  0.56  Aprenderás a mejorar tu labia para ligar, seducir a tu pareja o comunicarte con más fluidez en todas tus relaciones soci

Solo el clasificador embeddings: 84 libros. Algunas de sus oraciones más sospechosas:
  0.78  Libros para niñas y niños .
  0.95  Click the link in our profile to read the full novel without paying anything.
  0.79  Promoción exclusiva para nuevos usuarios: pagás uno y te llevás la saga entera.
  0.90  Inscribite ahora y obtené a

**Qué muestra**

- **406 sospechosos de 8.625 libros (4,7 %).** El catálogo limpio queda con 8.219.
- **El detector de idioma marcaría 64 libros más por su cuenta, y son falsos positivos:** oraciones en español con listas de nombres en inglés o escritas en mayúsculas («SE PRECISAN DOS DADOS DE 10 CARAS…» sale como alemán).
- **Solo el clasificador TF-IDF (38 libros):** en su mayoría falsos positivos. Aprendió que «tu» y «te» son señal de spam, porque los fragmentos le hablan al lector («Ganá», «Descargalo», «compartila»), y marca contratapas en segunda persona como «¡Tu bebé te lo agradecerá!». Pero también encuentra spam pegado a una oración legítima sin punto en el medio: «…un escritor mediocre de Si te gustó esta reseña…».
- **Solo el clasificador de embeddings (84 libros):** acá aparecen mensajes que no estaban en la muestra, en español y en inglés: «Promoción exclusiva para nuevos usuarios…», «Inscribite ahora y obtené acceso ilimitado…», «Join millions of readers…», «Click the link in our profile…». También hay falsos positivos, como «Libros para niñas y niños».
- **El costo de un falso positivo:** cada libro legítimo marcado sale del catálogo y nunca se va a recomendar. Por eso pedimos precisión 0,97 y dejamos afuera el detector de idioma. En una primera versión, con 0,90 y con el detector de idioma, marcábamos 694 libros: buena parte eran frases en segunda persona o títulos en inglés.

---
# Paso 3. Registrar las preferencias de un lector

**Lector de referencia:** _pendiente: el integrante del grupo que da todas las valoraciones_.

Mezclamos el catálogo limpio una sola vez y el lector recorre ese orden desde el principio. La primera vez, la celda de abajo crea la planilla `data/grupoNN_preferencias.csv` con los primeros 200 libros del orden y la columna `interes` vacía. El lector la completa de arriba hacia abajo, sin saltear libros, hasta reunir al menos 100 respuestas válidas y 20 de cada clase (ver el README):

| Respuesta del lector | `interes` |
|---|---|
| Sí, me interesaría leerlo | 1 |
| No me interesaría leerlo | 0 |
| No puede decidir, o el libro está contaminado | vacío |

In [28]:
limpio = pd.read_csv(RUTA_LIMPIO)
orden = limpio.sample(frac=1, random_state=SEMILLA)   # mezcla todo el catálogo; con la semilla, siempre el mismo orden

if not os.path.exists(RUTA_PREFERENCIAS):
    planilla = orden.head(200)[["slug", "title", "synopsis"]].assign(interes=pd.NA)
    planilla.to_csv(RUTA_PREFERENCIAS, index=False)
    print(f"Se creó {RUTA_PREFERENCIAS}: el lector tiene que completar la columna interes")

# Libros contaminados que encontró el lector y que el paso 2 no detectó: quedan con interes vacío
# y se agregan a los sospechosos con motivo "revision_manual". Completar con sus slugs
REVISION_MANUAL = []

Se creó data/grupoNN_preferencias.csv: el lector tiene que completar la columna interes


In [29]:
preferencias = pd.read_csv(RUTA_PREFERENCIAS)
preferencias["interes"] = preferencias["interes"].astype("Int64")   # entero que admite vacíos: 1, 0 o <NA>

# Las filas que siguen a la última respuesta son libros que el lector no llegó a ver: no son abstenciones
ultima = preferencias["interes"].last_valid_index()
if ultima is not None:
    preferencias = preferencias.loc[:ultima]

# Controles: el lector siguió el orden y los libros de revisión manual quedaron sin valorar
assert preferencias["slug"].tolist() == orden["slug"].head(len(preferencias)).tolist(), "la planilla no sigue el orden"
assert preferencias.loc[preferencias["slug"].isin(REVISION_MANUAL), "interes"].isna().all()

if REVISION_MANUAL:
    sospechosos = pd.read_csv(RUTA_SOSPECHOSOS)
    sospechosos = sospechosos[~sospechosos["slug"].isin(REVISION_MANUAL)]   # por si la celda se ejecuta dos veces
    manuales = pd.DataFrame({"slug": REVISION_MANUAL, "motivo": "revision_manual"})
    pd.concat([sospechosos, manuales]).to_csv(RUTA_SOSPECHOSOS, index=False)

conteo = preferencias["interes"].value_counts()
print(f"Libros vistos: {len(preferencias)}")
print(f"interes = 1: {conteo.get(1, 0)} · interes = 0: {conteo.get(0, 0)} · "
      f"sin valorar: {preferencias['interes'].isna().sum()} (de ellos, {len(REVISION_MANUAL)} por revisión manual)")
assert conteo.sum() >= 100 and conteo.get(1, 0) >= 20 and conteo.get(0, 0) >= 20, (
    f"Faltan valoraciones: completar la columna interes de {RUTA_PREFERENCIAS} (ver el README)")

Libros vistos: 200
interes = 1: 0 · interes = 0: 0 · sin valorar: 200 (de ellos, 0 por revisión manual)


AssertionError: Faltan valoraciones: completar la columna interes de data/grupoNN_preferencias.csv (ver el README)

> **Pendiente.** La planilla está creada y falta que el lector la complete: hasta entonces, el notebook se detiene en la celda anterior. Cuando esté completa, esta nota se reemplaza por cuántos libros vio el lector, cuántos 1 y cuántos 0 dio, y cuántas abstenciones hubo.

---
# Paso 4. Separar los libros para aprender y evaluar

Solo los libros con `interes` igual a 0 o 1: el 80 % para entrenamiento y el 20 % para prueba, con la misma división para todos los modelos.

In [ ]:
validas = preferencias.dropna(subset=["interes"]).copy()
validas["interes"] = validas["interes"].astype(int)
entrenamiento, prueba = train_test_split(validas, test_size=0.2, random_state=SEMILLA, stratify=validas["interes"])

# Columna conjunto: entrenamiento o prueba; los libros sin valorar quedan vacíos
preferencias["conjunto"] = pd.NA
preferencias.loc[entrenamiento.index, "conjunto"] = "entrenamiento"
preferencias.loc[prueba.index, "conjunto"] = "prueba"
preferencias.to_csv(RUTA_PREFERENCIAS, index=False)

pd.DataFrame({"libros": [len(entrenamiento), len(prueba)],
              "interes = 1": [entrenamiento["interes"].sum(), prueba["interes"].sum()]},
             index=["entrenamiento", "prueba"])

---
# Paso 5. Entrenar dos clasificadores

En los dos, la entrada es la sinopsis y la respuesta es `interes`, con `LogisticRegression(C=1, max_iter=1000, class_weight="balanced")`.

## Modelo A con TF-IDF

In [ ]:
y_entrenamiento = entrenamiento["interes"]
y_prueba = prueba["interes"]

# fit_transform aprende el vocabulario y los pesos solo con las sinopsis de entrenamiento;
# transform representa las de prueba con ese mismo vocabulario
tfidf = TfidfVectorizer()
X_entrenamiento = tfidf.fit_transform(entrenamiento["synopsis"])
X_prueba = tfidf.transform(prueba["synopsis"])

modelo_a = LogisticRegression(C=1, max_iter=1000, class_weight="balanced")
modelo_a.fit(X_entrenamiento, y_entrenamiento)
prediccion_a = modelo_a.predict(X_prueba)   # una predicción, 0 o 1, por cada libro de prueba
print(f"TF-IDF: entrenamiento {X_entrenamiento.shape}, prueba {X_prueba.shape}")
print("Predicciones:", prediccion_a)

## Modelo B con embeddings

Reutilizamos `multilingual-e5-small`, el modelo del paso 2.5, con el texto natural de cada sinopsis. El modelo de embeddings queda fijo: solo se entrena la regresión logística. Calculamos los embeddings de todo el catálogo limpio de una vez, porque el paso 8 los necesita; quedan guardados en `data/`.

In [ ]:
E_limpio = codificar_con_cache(limpio["synopsis"].tolist(), "data/embeddings_catalogo_limpio.npy")
fila = pd.Series(range(len(limpio)), index=limpio["slug"])   # slug → fila de E_limpio

E_entrenamiento = E_limpio[fila[entrenamiento["slug"]]]
E_prueba = E_limpio[fila[prueba["slug"]]]

modelo_b = LogisticRegression(C=1, max_iter=1000, class_weight="balanced")
modelo_b.fit(E_entrenamiento, y_entrenamiento)
prediccion_b = modelo_b.predict(E_prueba)
print(f"Embeddings: entrenamiento {E_entrenamiento.shape}, prueba {E_prueba.shape}")
print("Predicciones:", prediccion_b)

---
# Paso 6. Comprobar las predicciones

In [ ]:
PREDICCIONES = {"A · TF-IDF": prediccion_a, "B · embeddings": prediccion_b}
filas = {}
for nombre, prediccion in PREDICCIONES.items():
    print(f"=== Modelo {nombre}")
    print(classification_report(y_prueba, prediccion, zero_division=0))
    # Filas: valor real (0, 1); columnas: valor predicho (0, 1)
    print("Matriz de confusión:\n", confusion_matrix(y_prueba, prediccion), "\n")
    if prediccion.sum() == 0:
        print("El modelo no predijo ningún 1: precisión y F1 de la clase 1 quedan en 0 (zero_division=0)\n")
    p, r, f1, _ = precision_recall_fscore_support(y_prueba, prediccion, average="binary", zero_division=0)
    filas[nombre] = {"precisión": p, "recall": r, "F1": f1,
                     "aciertos": (prediccion == y_prueba).sum(), "errores": (prediccion != y_prueba).sum()}
pd.DataFrame(filas).T.astype({"aciertos": int, "errores": int}).round(2)

Con 20 libros de prueba, un solo libro cambia mucho las métricas. Para ver cuánto varían, una validación cruzada repetida sobre los libros de entrenamiento, con la misma configuración. En el modelo A el `Pipeline` ajusta el `TfidfVectorizer` dentro de cada partición. No la usamos para elegir parámetros.

In [ ]:
validacion = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEMILLA)
f1_a = cross_val_score(make_pipeline(TfidfVectorizer(), LogisticRegression(C=1, max_iter=1000, class_weight="balanced")),
                       entrenamiento["synopsis"], y_entrenamiento, cv=validacion, scoring="f1")
f1_b = cross_val_score(LogisticRegression(C=1, max_iter=1000, class_weight="balanced"),
                       E_entrenamiento, y_entrenamiento, cv=validacion, scoring="f1")
print(f"F1 de la clase 1 en 50 particiones (media ± desvío):")
print(f"  A · TF-IDF:     {f1_a.mean():.2f} ± {f1_a.std():.2f}")
print(f"  B · embeddings: {f1_b.mean():.2f} ± {f1_b.std():.2f}")

In [ ]:
# Los libros de prueba con su valoración real y lo que predijo cada modelo
casos = prueba[["title", "interes"]].assign(A=prediccion_a, B=prediccion_b)
casos["resultado A"] = np.select([casos["A"] == casos["interes"], casos["A"] == 1], ["acierto", "falso positivo"], "falso negativo")
casos["resultado B"] = np.select([casos["B"] == casos["interes"], casos["B"] == 1], ["acierto", "falso positivo"], "falso negativo")
casos

> **Pendiente (con las valoraciones).** Interpretar la precisión, el recall y el F1 de la clase 1 de cada modelo junto con los aciertos y errores, compararlos con la media y el desvío de la validación cruzada, y explicar qué pasó con dos libros de prueba, incluido un error si lo hubo.

---
# Paso 7. Convertir las predicciones en recomendaciones

Tres listas sobre los mismos libros de prueba: TF-IDF con clasificador, embeddings con clasificador y similitud con el perfil del lector (el promedio de los embeddings de los libros de entrenamiento con `interes` = 1).

In [ ]:
def probabilidad_interes(modelo, X):
    """Probabilidad estimada de interes = 1. classes_ dice qué columna de predict_proba es cada clase."""
    columna = list(modelo.classes_).index(1)
    return modelo.predict_proba(X)[:, columna]


print("classes_ de A:", modelo_a.classes_, "· classes_ de B:", modelo_b.classes_)

perfil = E_entrenamiento[y_entrenamiento.to_numpy() == 1].mean(axis=0)   # solo positivos de entrenamiento
puntajes_prueba = {"A · TF-IDF": probabilidad_interes(modelo_a, X_prueba),
                   "B · embeddings": probabilidad_interes(modelo_b, E_prueba),
                   "similitud con el perfil": cosine_similarity([perfil], E_prueba)[0]}

K = 5
azar = y_prueba.mean()   # proporción de 1 en prueba: la precisión esperada de elegir al azar
print(f"Referencia del azar: {azar:.2f}\n")
for nombre, puntajes in puntajes_prueba.items():
    lista = prueba[["title", "interes"]].assign(puntaje=puntajes.round(3))
    lista = lista.sort_values("puntaje", ascending=False).head(K)
    print(f"=== {nombre}: precision@{K} = {lista['interes'].sum() / K:.2f}")
    print(lista.to_string(index=False), "\n")

> **Pendiente (con las valoraciones).** Comparar la precision@5 de las tres listas con la referencia del azar y responder si los clasificadores recomendaron mejor que la similitud con el perfil.

---
# Paso 8. Recomendar libros del catálogo completo

Usamos el modelo B ya entrenado, sin volver a ajustarlo, sobre los libros del catálogo limpio que el lector no vio: se excluyen todos los de la planilla, también las abstenciones. Además, ninguna recomendación puede ser otra versión de un libro ya visto: mismos autores y mismo título según el criterio del paso 1.

In [ ]:
vistos = limpio[limpio["slug"].isin(preferencias["slug"])]
candidatos = limpio[~limpio["slug"].isin(preferencias["slug"])].copy()
candidatos["puntaje"] = probabilidad_interes(modelo_b, E_limpio[fila[candidatos["slug"]]])


def es_otra_version(libro):
    """True si el libro tiene los mismos autores y el mismo título que alguno de los vistos."""
    mismos_autores = vistos[vistos["authors"] == libro["authors"]]
    return any(mismo_titulo(libro["title"], titulo) for titulo in mismos_autores["title"])


candidatos["otra_version"] = candidatos.apply(es_otra_version, axis=1)
print(f"Candidatos: {len(candidatos)}; otras versiones de libros vistos (se descartan): {candidatos['otra_version'].sum()}")
candidatos = candidatos[~candidatos["otra_version"]].sort_values("puntaje", ascending=False)

for _, libro in candidatos.head(5).iterrows():
    print(f"\n{libro['title']} — {libro['authors']} (puntaje {libro['puntaje']:.3f})\n{libro['synopsis']}")

**Comentario del lector sobre las cinco recomendaciones:** _pendiente: el lector indica cuáles le interesan y cuáles no._

## Resumir las recomendaciones

Resumen extractivo de dos oraciones con el método de la Unidad 3: las oraciones se representan con TF-IDF y se conservan las dos más centrales (mayor similitud media con las demás), en su orden original.

In [ ]:
def resumir(texto, k=2):
    """Las k oraciones más centrales del texto, en su orden original."""
    frases = [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto) if f.strip()]
    if len(frases) <= k:
        return texto
    sim = cosine_similarity(TfidfVectorizer().fit_transform(frases))
    np.fill_diagonal(sim, 0)   # la similitud de cada oración consigo misma (1) no cuenta
    centralidad = sim.mean(axis=1)
    elegidas = np.sort(np.argsort(centralidad)[-k:])   # las k más centrales, ordenadas por posición en el texto
    return " ".join(frases[i] for i in elegidas)


for _, libro in candidatos.head(5).iterrows():
    print(f"{libro['title']} ({len(libro['synopsis'])} → {len(resumir(libro['synopsis']))} caracteres)\n   {resumir(libro['synopsis'])}\n")

> **Pendiente (con las recomendaciones).** Mostrar un caso en el que el resumen funcionó bien y uno en el que perdió información importante.

## Describir el perfil del lector

Frases sustantivas (`noun_chunks`) y entidades de las sinopsis de entrenamiento, separadas por `interes`. Las frases se agrupan por el lema de su núcleo («las novelas», «una novela» → *novela*); nos quedamos con los núcleos que son sustantivos comunes, porque los pronombres («él», «que») no describen gustos.

In [ ]:
def perfil_de(textos):
    """Las diez frases sustantivas (por el lema de su núcleo) y las diez entidades más frecuentes."""
    nucleos, entidades = Counter(), Counter()
    for doc in nlp.pipe(textos):
        nucleos.update(chunk.root.lemma_.lower() for chunk in doc.noun_chunks if chunk.root.pos_ == "NOUN")
        entidades.update(ent.text for ent in doc.ents)
    return [f"{n} ({c})" for n, c in nucleos.most_common(10)], [f"{e} ({c})" for e, c in entidades.most_common(10)]


frases_1, entidades_1 = perfil_de(entrenamiento.loc[y_entrenamiento == 1, "synopsis"])
frases_0, entidades_0 = perfil_de(entrenamiento.loc[y_entrenamiento == 0, "synopsis"])
pd.DataFrame({"frases, interes = 1": frases_1, "frases, interes = 0": frases_0,
              "entidades, interes = 1": entidades_1, "entidades, interes = 0": entidades_0}, index=range(1, 11))

**Comentario del lector sobre el perfil:** _pendiente: el lector comenta si estas listas describen lo que le gusta._

## La función `recomendar`

In [ ]:
def recomendar(n=5):
    """Los n libros no vistos con mayor probabilidad de interés según el modelo B, con su resumen."""
    elegidos = candidatos.head(n)   # candidatos ya está ordenado por puntaje y sin otras versiones
    return pd.DataFrame({"slug": elegidos["slug"], "title": elegidos["title"],
                         "puntaje": elegidos["puntaje"].round(3),
                         "resumen": elegidos["synopsis"].apply(resumir)}).reset_index(drop=True)


recomendar()

> Una puntuación alta es una estimación del modelo, no una garantía de que el libro le va a gustar al lector después de leerlo. Los comentarios del lector sobre estas recomendaciones quedan separados de las métricas de prueba.